# PyTorch 神经推荐架构复现：Two-Tower 与 NeuMF

这本 Notebook 不调用现成推荐模型，而是用基础 \`nn.Embedding\`、\`nn.Linear\` 和张量运算实现两个常见架构：

1. \`TwoTowerRecommender\`：用户塔与物品内容塔分别编码，适合离线建 item embedding 和候选召回；
2. \`NeuMF\`：把矩阵分解式逐元素交互与 MLP 交互拼接，适合讲清 Neural Collaborative Filtering 的 forward。

重点不只是把类写出来，还包括时间切分、曝光负例、BPR/BCE、全候选 ranking 指标、主体权限绑定、state_dict 指纹和失败反例。

> 数据高度规则化，目标是验证架构、梯度和系统合同。小样本高分不能外推真实推荐效果。

## 1. 运行边界与可复现性

Notebook 默认 CPU、固定 Python/NumPy/PyTorch 随机种子，不下载数据。CUDA 可用于真实训练，但不能成为本示例的隐式依赖。训练产物必须同时绑定特征 schema、catalog generation、切分 cutoff 和代码版本；只保存一份权重无法复现推荐。

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="The pynvml package is deprecated", category=FutureWarning)

from copy import deepcopy
from dataclasses import asdict, dataclass, field
from hashlib import sha256
import json
import math
import random
import numpy as np
import torch
from torch import nn

SEED = 2801
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
DEVICE = torch.device("cpu")

assert torch.__version__
assert DEVICE.type == "cpu"
print({"torch": torch.__version__, "device": str(DEVICE), "seed": SEED})

## 2. 时间化隐式反馈合同

目标是“在 round 5 推荐用户将交互的 item”。每条训练样本包含同一曝光请求中的正例与明确曝光后跳过的负例，而不是把所有未观察 item 都当真负例。

按全局 round 切分：

- train：0–3；
- validation：4；
- test：5。

用户跨 split 出现表示“已有用户未来推荐”。如果目标是新用户冷启动，必须另建 user-disjoint 测试。item 内容特征由类目 one-hot 与质量组成，并在交互前已知。

In [ ]:
N_USERS, N_ITEMS, N_CATEGORIES = 12, 32, 4

@dataclass(frozen=True)
class ExposurePair:
    event_id: str
    user_id: int
    positive_item: int
    exposed_negative: int
    round_id: int
    tenant: str = "tenant-a"

item_category = torch.arange(N_ITEMS) // 8
item_quality = (torch.arange(N_ITEMS) % 8).float() / 7
item_features = torch.cat([
    torch.nn.functional.one_hot(item_category, N_CATEGORIES).float(),
    item_quality[:, None],
], dim=1)

preferences = {u: (u % 4, (u + 1) % 4) for u in range(N_USERS)}
events = []
for round_id in range(6):
    for user in range(N_USERS):
        category = preferences[user][round_id % 2]
        offset = (user // 4 + round_id) % 8
        positive = category * 8 + offset
        negative_category = (category + 2) % 4
        negative = negative_category * 8 + offset
        events.append(ExposurePair(
            f"exp-{round_id:02d}-{user:02d}", user, positive, negative, round_id
        ))

train_events = [e for e in events if e.round_id <= 3]
valid_events = [e for e in events if e.round_id == 4]
test_events = [e for e in events if e.round_id == 5]
assert len(events) == 72
assert len({e.event_id for e in events}) == len(events)
assert max(e.round_id for e in train_events) < min(e.round_id for e in valid_events)
assert max(e.round_id for e in valid_events) < min(e.round_id for e in test_events)
assert item_features.shape == (N_ITEMS, N_CATEGORIES + 1)
print({"train": len(train_events), "validation": len(valid_events), "test": len(test_events)})

## 3. 负例必须带来源

BPR 的三元组 $(u,i,j)$ 假设 $i$ 比 $j$ 更受偏好，但“未点击”只有在 item 真正曝光后才更接近负反馈。这里使用日志里的 \`exposed_negative\`。同时审计一种常见弱方案：只排除 train positive 的随机未观察采样可能撞上 validation/test 的未来正例；这个碰撞率只能用于诊断，不能用未来标签修训练采样。

In [ ]:
train_positive = {u: set() for u in range(N_USERS)}
future_positive = {u: set() for u in range(N_USERS)}
for event in train_events:
    train_positive[event.user_id].add(event.positive_item)
for event in valid_events + test_events:
    future_positive[event.user_id].add(event.positive_item)

audit_rng = np.random.default_rng(SEED)
weak_samples = []
for user in range(N_USERS):
    pool = [item for item in range(N_ITEMS) if item not in train_positive[user]]
    weak_samples.append((user, int(audit_rng.choice(pool))))
future_collision = np.mean([item in future_positive[user] for user, item in weak_samples])

assert all(e.exposed_negative != e.positive_item for e in train_events)
assert all(item not in train_positive[user] for user, item in weak_samples)
assert 0.0 <= future_collision <= 1.0
print({"weak_unobserved_future_collision": float(future_collision)})

## 4. Two-Tower：可拆分的召回架构

用户塔把 user ID 映射为向量；物品塔只读发布前可得的内容特征，因此新 item 即使没有交互，也能产生 embedding。两个输出做 L2 归一化，分数是余弦相似度除以温度：

$$s(u,i)=\frac{\bar z_u^\top \bar z_i}{\tau}.$$

生产系统会把 item 塔离线批量计算后写入 ANN 索引；用户塔在线计算。两塔的 preprocess、维度、归一化和距离必须版本一致。

In [ ]:
class TwoTowerRecommender(nn.Module):
    def __init__(self, n_users, item_feature_dim, hidden_dim=16, embedding_dim=8, temperature=0.2):
        super().__init__()
        if temperature <= 0:
            raise ValueError("temperature_must_be_positive")
        self.user_embedding = nn.Embedding(n_users, hidden_dim)
        self.user_tower = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, embedding_dim),
        )
        self.item_tower = nn.Sequential(
            nn.Linear(item_feature_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, embedding_dim),
        )
        self.n_users = int(n_users)
        self.item_feature_dim = int(item_feature_dim)
        self.hidden_dim = int(hidden_dim)
        self.embedding_dim = int(embedding_dim)
        self.temperature = float(temperature)

    def encode_user(self, user_ids):
        raw = self.user_tower(self.user_embedding(user_ids))
        return torch.nn.functional.normalize(raw, dim=-1)

    def encode_item(self, features):
        raw = self.item_tower(features)
        return torch.nn.functional.normalize(raw, dim=-1)

    def forward(self, user_ids, features):
        if user_ids.ndim != 1 or features.ndim != 2 or len(user_ids) != len(features):
            raise ValueError("batch_shape_mismatch")
        return (self.encode_user(user_ids) * self.encode_item(features)).sum(-1) / self.temperature

two_tower = TwoTowerRecommender(N_USERS, item_features.shape[1]).to(DEVICE)
probe_users = torch.tensor([0, 1, 2])
probe_scores = two_tower(probe_users, item_features[[0, 8, 16]])
assert probe_scores.shape == (3,)
assert torch.isfinite(probe_scores).all()
assert torch.allclose(two_tower.encode_item(item_features[:4]).norm(dim=1), torch.ones(4), atol=1e-5)

## 5. NeuMF：显式组合 GMF 与 MLP 路径

NeuMF 同时学习：

- GMF：user/item embedding 的逐元素乘积；
- MLP：拼接另一组 user/item embedding 后经过非线性层；
- 最终 head：拼接两条路径并输出一个 logit。

这里不调用任何推荐库，所有参数和 forward 都可见。它依赖 item ID embedding，因此对从未训练的新 item 没有 Two-Tower 内容塔那样的自然冷启动能力。

In [ ]:
class NeuMF(nn.Module):
    def __init__(self, n_users, n_items, factor_dim=8):
        super().__init__()
        self.user_gmf = nn.Embedding(n_users, factor_dim)
        self.item_gmf = nn.Embedding(n_items, factor_dim)
        self.user_mlp = nn.Embedding(n_users, factor_dim)
        self.item_mlp = nn.Embedding(n_items, factor_dim)
        self.mlp = nn.Sequential(
            nn.Linear(2 * factor_dim, 2 * factor_dim), nn.ReLU(),
            nn.Linear(2 * factor_dim, factor_dim), nn.ReLU(),
        )
        self.output = nn.Linear(2 * factor_dim, 1)

    def forward(self, user_ids, item_ids):
        if user_ids.shape != item_ids.shape:
            raise ValueError("user_item_shape_mismatch")
        gmf = self.user_gmf(user_ids) * self.item_gmf(item_ids)
        mlp_input = torch.cat([self.user_mlp(user_ids), self.item_mlp(item_ids)], dim=-1)
        mlp_output = self.mlp(mlp_input)
        return self.output(torch.cat([gmf, mlp_output], dim=-1)).squeeze(-1)

neumf = NeuMF(N_USERS, N_ITEMS).to(DEVICE)
assert neumf(torch.tensor([0, 1]), torch.tensor([1, 2])).shape == (2,)
try:
    neumf(torch.tensor([0, 1]), torch.tensor([1]))
    raise AssertionError("shape mismatch must fail")
except ValueError:
    pass

## 6. BPR 与 BCE 的目标不同

Two-Tower 用曝光 pair 训练 BPR：

$$L=-\log \sigma(s(u,i)-s(u,j)).$$

NeuMF 把正负曝光展开成二分类样本，用 \`BCEWithLogitsLoss\`。BPR 直接关心相对次序；BCE 的概率含义还受采样率影响，不能把训练 batch 的 sigmoid 当无偏点击率。

In [ ]:
def bpr_loss(positive_score, negative_score):
    if positive_score.shape != negative_score.shape:
        raise ValueError("pair_shape_mismatch")
    return -torch.nn.functional.logsigmoid(positive_score - negative_score).mean()

margin = torch.tensor([2.0, -1.0], requires_grad=True)
loss_probe = bpr_loss(margin, torch.zeros_like(margin))
loss_probe.backward()
assert loss_probe.item() > 0
assert margin.grad is not None and torch.isfinite(margin.grad).all()
assert margin.grad[0] < 0 and margin.grad[1] < 0

training_event_ids = tuple(e.event_id for e in train_events)
train_users = torch.tensor([e.user_id for e in train_events], dtype=torch.long)
train_pos = torch.tensor([e.positive_item for e in train_events], dtype=torch.long)
train_neg = torch.tensor([e.exposed_negative for e in train_events], dtype=torch.long)
assert train_users.shape == train_pos.shape == train_neg.shape
assert set(training_event_ids) == {e.event_id for e in train_events}
assert set(training_event_ids).isdisjoint(e.event_id for e in valid_events + test_events)

## 7. 受控训练：只优化 train

两个模型都只读取 round 0–3。validation/test 不参与梯度，也不用于挑 epoch；本例预先固定训练步数，目的是证明参数能通过 forward/backward 学习这个受控模式。真实项目应使用 validation 选择 checkpoint，再冻结 test 只报告一次，并记录 sampler、optimizer、随机种子和曝光策略版本。

In [ ]:
optimizer = torch.optim.Adam(two_tower.parameters(), lr=0.03)
two_losses = []
for step in range(220):
    optimizer.zero_grad(set_to_none=True)
    positive = two_tower(train_users, item_features[train_pos])
    negative = two_tower(train_users, item_features[train_neg])
    loss = bpr_loss(positive, negative)
    loss.backward()
    if step == 0:
        initial_grad_norm = torch.sqrt(sum(
            (p.grad.detach() ** 2).sum() for p in two_tower.parameters() if p.grad is not None
        ))
    torch.nn.utils.clip_grad_norm_(two_tower.parameters(), 5.0)
    optimizer.step()
    two_losses.append(float(loss.detach()))

assert math.isfinite(two_losses[-1])
assert two_losses[-1] < two_losses[0] * 0.2
assert initial_grad_norm > 0 and torch.isfinite(initial_grad_norm)
print({"two_tower_loss": [round(two_losses[0], 4), round(two_losses[-1], 4)]})

In [ ]:
bce_users = torch.cat([train_users, train_users])
bce_items = torch.cat([train_pos, train_neg])
bce_labels = torch.cat([torch.ones(len(train_pos)), torch.zeros(len(train_neg))])
bce_optimizer = torch.optim.Adam(neumf.parameters(), lr=0.025)
bce_loss_fn = nn.BCEWithLogitsLoss()
ncf_losses = []
for step in range(180):
    bce_optimizer.zero_grad(set_to_none=True)
    logits = neumf(bce_users, bce_items)
    ncf_loss = bce_loss_fn(logits, bce_labels)
    ncf_loss.backward()
    bce_optimizer.step()
    ncf_losses.append(float(ncf_loss.detach()))

with torch.no_grad():
    train_accuracy = ((torch.sigmoid(neumf(bce_users, bce_items)) >= 0.5) == bce_labels.bool()).float().mean()
assert ncf_losses[-1] < ncf_losses[0] * 0.25
assert train_accuracy >= 0.95
print({"neumf_loss": [round(ncf_losses[0], 4), round(ncf_losses[-1], 4)],
       "fixture_train_accuracy": float(train_accuracy)})

## 8. rolling round 5 的全 catalog 排名评估

本例明确采用 **rolling one-step** 协议：round 4 的 validation 交互发生后，才为 round 5 产生推荐。因此 validation 标签不参与梯度或模型选择之外的训练，但在 round 5 已是可见历史，候选集合必须排除 train + validation 的已交互 item。若业务允许重复消费，应另写显式策略，不能悄悄把旧 item 留在候选中。

Two-Tower 与 NeuMF 都对全部 32 个 item 打分，再做历史过滤；没有 sampled-negative 捷径。两者都在同一 held-out test gold 上报告 Recall@10、MRR@10、nDCG@10 与 coverage@10。真实系统还要分开看 candidate recall 与 final rank、长尾、新用户、新 item、群体差异和曝光偏差。

In [ ]:
@torch.no_grad()
def rank_all_items(model, user, excluded):
    model.eval()
    item_ids = torch.arange(N_ITEMS, dtype=torch.long)
    user_ids = torch.full((N_ITEMS,), user, dtype=torch.long)
    if isinstance(model, TwoTowerRecommender):
        scores = model(user_ids, item_features).cpu().numpy()
    elif isinstance(model, NeuMF):
        scores = model(user_ids, item_ids).cpu().numpy()
    else:
        raise TypeError("unsupported_recommender")
    allowed = [item for item in range(N_ITEMS) if item not in excluded]
    return sorted(allowed, key=lambda item: (-float(scores[item]), item))

def ranking_metrics(ranking, relevant, k=10):
    hits = [rank for rank, item in enumerate(ranking[:k], 1) if item in relevant]
    recall = len(set(ranking[:k]) & relevant) / len(relevant) if relevant else 0.0
    reciprocal_rank_at_k = 1 / hits[0] if hits else 0.0
    dcg = sum(1 / math.log2(rank + 1) for rank, item in enumerate(ranking[:k], 1) if item in relevant)
    ideal = sum(1 / math.log2(rank + 1) for rank in range(1, min(k, len(relevant)) + 1))
    return recall, reciprocal_rank_at_k, dcg / ideal if ideal else 0.0

validation_positive = {
    user: {e.positive_item for e in valid_events if e.user_id == user}
    for user in range(N_USERS)
}
serving_history = {
    user: train_positive[user] | validation_positive[user]
    for user in range(N_USERS)
}
test_gold = {
    user: {e.positive_item for e in test_events if e.user_id == user}
    for user in range(N_USERS)
}
assert all(validation_positive[user] <= serving_history[user] for user in range(N_USERS))
assert all(validation_positive[user].isdisjoint(train_positive[user]) for user in range(N_USERS))

def evaluate_full_catalog(model, excluded_history, k=10):
    rows, top_items = [], set()
    for user in range(N_USERS):
        ranking = rank_all_items(model, user, excluded_history[user])
        assert len(ranking) == N_ITEMS - len(excluded_history[user])
        assert set(ranking).isdisjoint(excluded_history[user])
        rows.append(ranking_metrics(ranking, test_gold[user], k))
        top_items.update(ranking[:k])
    return np.mean(rows, axis=0), len(top_items) / N_ITEMS

two_tower_metrics, two_tower_coverage = evaluate_full_catalog(two_tower, serving_history)
neumf_metrics, neumf_coverage = evaluate_full_catalog(neumf, serving_history)
heldout_metrics = {
    "TwoTower": {"Recall@10": two_tower_metrics[0], "MRR@10": two_tower_metrics[1],
                  "nDCG@10": two_tower_metrics[2], "coverage@10": two_tower_coverage},
    "NeuMF": {"Recall@10": neumf_metrics[0], "MRR@10": neumf_metrics[1],
               "nDCG@10": neumf_metrics[2], "coverage@10": neumf_coverage},
}
offline_metrics = two_tower_metrics
catalog_coverage = two_tower_coverage
print(heldout_metrics)
assert np.isfinite(two_tower_metrics).all() and np.isfinite(neumf_metrics).all()
assert all(0.0 <= value <= 1.0 for value in np.r_[two_tower_metrics, neumf_metrics])
assert two_tower_metrics[0] >= 0.75
assert 0.0 < two_tower_coverage <= 1.0 and 0.0 < neumf_coverage <= 1.0

## 9. 参数量、梯度与冷启动边界

参数量不是模型文件的全部成本：Two-Tower 还需要全量 item embedding、ANN 索引和在线 user tower；NeuMF 需要对每个候选运行交互网络。ID embedding 的容量随实体数线性增长。

新 item 可以通过 Two-Tower 内容特征直接编码；新 user 没有 embedding 时必须路由热门/上下文模型，不能随意取 user 0。任何未知或越界 ID 都要在接口层拒绝或显式降级。

In [ ]:
def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

two_parameter_count = count_parameters(two_tower)
ncf_parameter_count = count_parameters(neumf)
assert two_parameter_count > 0 and ncf_parameter_count > 0
assert two_tower.encode_item(item_features[[0]]).shape == (1, 8)

new_item_features = torch.tensor([[0.0, 1.0, 0.0, 0.0, 0.5]])
new_item_embedding = two_tower.encode_item(new_item_features)
assert new_item_embedding.shape == (1, 8)
assert torch.allclose(new_item_embedding.norm(dim=1), torch.ones(1), atol=1e-5)

try:
    two_tower(torch.tensor([N_USERS]), item_features[[0]])
    raise AssertionError("unknown user must fail")
except (IndexError, RuntimeError):
    pass
print({"two_tower_params": two_parameter_count, "neumf_params": ncf_parameter_count})

## 10. 服务、权限与可执行的制品绑定

推荐请求不能让客户端自由指定任意 user ID。认证层签发 tenant 与 subject，服务把 subject 映射到内部 user ID，防止 IDOR 读取他人的个性化结果。召回前过滤 tenant、上下架、库存与合规；排序后再过滤会浪费 top-K 并可能泄漏。

这里不只是“把 hash 写进字典”：模型必须先注册到进程内受信 registry，每次推荐前都调用 `validate_artifact`，重算 state_dict、item feature/catalog、事件快照与 rolling history 指纹，并核对模型配置和 bundle。任何权重、特征、历史或版本漂移都 fail closed；即便伪造模型拥有相同结构和权重，也因对象不在 registry 而被拒绝。hash 仍不是签名，生产还要用签名制品、只读对象仓库、代码镜像和可回滚 catalog generation。

In [ ]:
_AUTH_MARKER = object()

@dataclass(frozen=True)
class AuthContext:
    tenant: str
    subject: str
    _marker: object = field(repr=False, compare=False)

def authenticate_demo(token):
    if token != "signed-user-0":
        raise PermissionError("authentication_failed")
    return AuthContext("tenant-a", "user-0", _AUTH_MARKER)

def tensor_sha256(tensor):
    value = tensor.detach().cpu().contiguous()
    digest = sha256()
    digest.update(str(value.dtype).encode())
    digest.update(json.dumps(list(value.shape)).encode())
    digest.update(value.numpy().tobytes())
    return digest.hexdigest()

def state_dict_sha256(model):
    digest = sha256()
    for name, tensor in sorted(model.state_dict().items()):
        digest.update(name.encode())
        digest.update(tensor_sha256(tensor).encode())
    return digest.hexdigest()

def event_log_sha256(event_log):
    payload = [asdict(event) for event in event_log]
    return sha256(json.dumps(payload, ensure_ascii=False, sort_keys=True).encode()).hexdigest()

def history_sha256(history):
    payload = {str(user): sorted(items) for user, items in sorted(history.items())}
    return sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()

def artifact_bundle_sha256(artifact):
    payload = {key: value for key, value in artifact.items() if key != "bundle_sha256"}
    return sha256(json.dumps(payload, ensure_ascii=False, sort_keys=True).encode()).hexdigest()

def model_config(model):
    return {
        "n_users": model.n_users, "item_feature_dim": model.item_feature_dim,
        "hidden_dim": model.hidden_dim, "embedding_dim": model.embedding_dim,
        "temperature": model.temperature,
    }

def history_through_round(event_log, round_max):
    history = {user: set() for user in range(N_USERS)}
    for event in event_log:
        if event.round_id <= round_max:
            history[event.user_id].add(event.positive_item)
    return history

def validate_runtime_payload(model, artifact, features, event_log, history):
    if type(model) is not TwoTowerRecommender:
        raise TypeError("artifact_architecture_mismatch")
    if artifact_bundle_sha256(artifact) != artifact.get("bundle_sha256"):
        raise RuntimeError("artifact_bundle_hash_mismatch")
    if artifact.get("architecture") != type(model).__name__ or artifact.get("model_config") != model_config(model):
        raise RuntimeError("artifact_model_config_mismatch")
    if artifact.get("state_dict_sha256") != state_dict_sha256(model):
        raise RuntimeError("artifact_state_dict_mismatch")
    if artifact.get("item_features_sha256") != tensor_sha256(features):
        raise RuntimeError("artifact_catalog_feature_mismatch")
    if artifact.get("event_log_sha256") != event_log_sha256(event_log):
        raise RuntimeError("artifact_event_snapshot_mismatch")
    expected_history = history_through_round(event_log, artifact["validation_round_max"])
    if history != expected_history or artifact.get("serving_history_sha256") != history_sha256(history):
        raise RuntimeError("artifact_serving_history_mismatch")
    if features.shape != (artifact["catalog_size"], artifact["model_config"]["item_feature_dim"]):
        raise RuntimeError("artifact_catalog_shape_mismatch")
    if not torch.isfinite(features).all():
        raise RuntimeError("artifact_nonfinite_features")
    return True

manifest = {
    "architecture": "TwoTowerRecommender",
    "model_version": "two-tower-v1",
    "code_version": "notebook-28-contract-v2",
    "tenant": "tenant-a",
    "model_config": model_config(two_tower),
    "feature_schema": ["category_0", "category_1", "category_2", "category_3", "quality"],
    "catalog_size": N_ITEMS,
    "catalog_generation": "catalog-g1",
    "train_round_max": max(e.round_id for e in train_events),
    "validation_round_max": max(e.round_id for e in valid_events),
    "item_features_sha256": tensor_sha256(item_features),
    "event_log_sha256": event_log_sha256(events),
    "serving_history_sha256": history_sha256(serving_history),
    "state_dict_sha256": state_dict_sha256(two_tower),
}
manifest["bundle_sha256"] = artifact_bundle_sha256(manifest)
_TRUSTED_MODEL_REGISTRY = {}

def register_trusted_artifact(model, artifact, features, event_log, history):
    validate_runtime_payload(model, artifact, features, event_log, history)
    version = artifact["model_version"]
    if version in _TRUSTED_MODEL_REGISTRY:
        raise RuntimeError("model_version_already_registered")
    _TRUSTED_MODEL_REGISTRY[version] = {"model": model, "artifact": deepcopy(artifact)}

def validate_artifact(model, artifact, features, event_log, history):
    trusted = _TRUSTED_MODEL_REGISTRY.get(artifact.get("model_version"))
    if trusted is None or trusted["model"] is not model:
        raise PermissionError("untrusted_model")
    if trusted["artifact"] != artifact:
        raise PermissionError("untrusted_artifact")
    return validate_runtime_payload(model, artifact, features, event_log, history)

register_trusted_artifact(two_tower, manifest, item_features, events, serving_history)

def recommend(user_id, auth, top_k=5, model_version="two-tower-v1"):
    if not isinstance(auth, AuthContext) or auth._marker is not _AUTH_MARKER:
        raise PermissionError("untrusted_auth")
    trusted = _TRUSTED_MODEL_REGISTRY.get(model_version)
    if trusted is None:
        raise PermissionError("untrusted_model_version")
    model, artifact = trusted["model"], trusted["artifact"]
    validate_artifact(model, artifact, item_features, events, serving_history)
    if auth.tenant != artifact["tenant"] or auth.subject != f"user-{user_id}":
        raise PermissionError("subject_user_mismatch")
    if not isinstance(user_id, int) or not 0 <= user_id < artifact["model_config"]["n_users"]:
        raise ValueError("invalid_user_id")
    if not 1 <= top_k <= 20:
        raise ValueError("invalid_top_k")
    ranking = rank_all_items(model, user_id, serving_history[user_id])[:top_k]
    return ranking, {"model_version": artifact["model_version"],
                     "catalog_generation": artifact["catalog_generation"],
                     "bundle_sha256": artifact["bundle_sha256"]}

AUTH = authenticate_demo("signed-user-0")
served, trace = recommend(0, AUTH)
assert served and len(manifest["state_dict_sha256"]) == 64
assert len(manifest["bundle_sha256"]) == 64
assert validate_artifact(two_tower, _TRUSTED_MODEL_REGISTRY["two-tower-v1"]["artifact"], item_features, events, serving_history)

## 11. 研究来源与生产边界

- Rendle et al., *BPR: Bayesian Personalized Ranking from Implicit Feedback*, UAI 2009：https://arxiv.org/abs/1205.2618
- He et al., *Neural Collaborative Filtering*, WWW 2017：https://doi.org/10.1145/3038912.3052569
- Yi et al., *Sampling-Bias-Corrected Neural Modeling for Large Corpus Item Recommendations*, RecSys 2019：https://doi.org/10.1145/3298689.3346996
- PyTorch 官方文档，\`nn.Module\`、\`Embedding\` 与 autograd：https://pytorch.org/docs/stable/nn.html

本例没有真实曝光位置、序列兴趣、ANN、特征平台、反事实估计或在线实验。架构复现只证明 forward、loss、梯度、受控 ranking 与服务合同能协同工作。

In [ ]:
# 最终回归：时间、架构、梯度、全目录排名、权限与 fail-closed 制品
assert max(e.round_id for e in train_events) == 3
assert min(e.round_id for e in valid_events) == 4
assert min(e.round_id for e in test_events) == 5
assert set(training_event_ids).isdisjoint(e.event_id for e in valid_events)
assert all(e.positive_item in serving_history[e.user_id] for e in valid_events)
assert two_tower(probe_users, item_features[[0, 8, 16]]).shape == (3,)
assert torch.isfinite(torch.stack([p.detach().abs().mean() for p in two_tower.parameters()])).all()
assert two_losses[-1] < two_losses[0]
assert ncf_losses[-1] < ncf_losses[0]
assert heldout_metrics["TwoTower"]["Recall@10"] >= 0.75
assert set(heldout_metrics) == {"TwoTower", "NeuMF"}
assert all(item not in serving_history[0] for item in served)
assert manifest["feature_schema"][-1] == "quality"
assert manifest["item_features_sha256"] == tensor_sha256(item_features)
assert trace["bundle_sha256"] == manifest["bundle_sha256"]

try:
    recommend(1, AUTH)
    raise AssertionError("IDOR must fail")
except PermissionError:
    pass
try:
    recommend(0, AUTH, top_k=0)
    raise AssertionError("invalid top_k must fail")
except ValueError:
    pass

trusted_artifact = _TRUSTED_MODEL_REGISTRY["two-tower-v1"]["artifact"]
parameter = next(two_tower.parameters())
parameter_backup = parameter.detach().clone()
try:
    with torch.no_grad():
        parameter.add_(0.125)
    try:
        recommend(0, AUTH)
        raise AssertionError("tampered weights must fail closed")
    except RuntimeError as error:
        assert str(error) == "artifact_state_dict_mismatch"
finally:
    with torch.no_grad():
        parameter.copy_(parameter_backup)

feature_backup = item_features[0, 0].item()
try:
    item_features[0, 0] += 0.25
    try:
        recommend(0, AUTH)
        raise AssertionError("tampered catalog features must fail closed")
    except RuntimeError as error:
        assert str(error) == "artifact_catalog_feature_mismatch"
finally:
    item_features[0, 0] = feature_backup

events.append(ExposurePair("tampered-event", 0, 0, 16, 0))
try:
    try:
        recommend(0, AUTH)
        raise AssertionError("tampered data snapshot must fail closed")
    except RuntimeError as error:
        assert str(error) == "artifact_event_snapshot_mismatch"
finally:
    events.pop()

forged_model = TwoTowerRecommender(N_USERS, item_features.shape[1]).to(DEVICE)
forged_model.load_state_dict(two_tower.state_dict())
try:
    validate_artifact(forged_model, trusted_artifact, item_features, events, serving_history)
    raise AssertionError("unregistered model must fail closed")
except PermissionError as error:
    assert str(error) == "untrusted_model"

tampered_artifact = deepcopy(trusted_artifact)
tampered_artifact["catalog_generation"] = "attacker-catalog"
tampered_artifact["bundle_sha256"] = artifact_bundle_sha256(tampered_artifact)
try:
    validate_artifact(two_tower, tampered_artifact, item_features, events, serving_history)
    raise AssertionError("untrusted artifact must fail closed")
except PermissionError as error:
    assert str(error) == "untrusted_artifact"

assert validate_artifact(two_tower, trusted_artifact, item_features, events, serving_history)
assert recommend(0, AUTH)[0]
print("神经推荐架构、双模型全目录评估、rolling history、权限和可信制品回归全部通过。")